# Birdcall Classifier Project

## What is it? 
This classifier categorizes birdcalls from a labelled dataset of four bird species. With the input of a .wav file containing a potential birdcall, the classifier builds a decision tree from the training set and outputs predictions from a test set to a csv file with approximately 80% accuracy. 

## Why?
Feature extraction is a vital part of data science and machine learning workflows. To practice feature extraction that can effectively classify bird calls, I limited myself to only classical machine learning methods (kNN, SVM etc) to challenge myself to carefully consider feature extraction. 
<img src="https://upload.wikimedia.org/wikipedia/commons/0/03/201903_Zebra_finch.svg" width="250">
## Dataset
The dataset consists of 3 second audio samples extracted from longer recordings and then manually annotated, featuring a variety of bird species and environmental conditions. The labels: {0, 1, 2, 3, 4} correspond to {American_Robin, Carolina_Wren, Northern_Cardinal, Song Sparrow, White-crowned Sparrow}. Some samples are of very poor quality, containing a lot of noise or environmental background. The dataset is copyright of Jayadev Acharya, a professor at Cornell, and used with permission. 

## Data Processing/Feature Extraction:
Feature extraction was done via the Librosa package. Librosa is a package for audio analysis, and thus applicable to the task at hand. I experimented with various combinations of features and their location in the input array to achieve the maximum possible accuracy. The features were flattened and concatenated into a 1d array for model training in the following order. A brief description of each follows below.

### Librosa.feature.mfcc:
Mel-frequency cepstral coefficients (MFCCs) are features widely used in audio analysis and speech recognition to represent the spectral properties of a sound signal in a way that mimics human auditory perception. MFCCs are computed using the Mel scale, which is a nonlinear scale that approximates the way humans perceive pitch and frequency. On the Mel scale, lower frequencies are more finely resolved than higher frequencies. By selecting the features most relevant for human birdcall perception, the MFCCs increase the signal to noise ratio of the feature space and with it model accuracy.

### Librosa.feature.rms: 
Root-mean-square (RMS) values are another statistical measure of center in audio analysis. RMS values provide a way to summarize the overall amplitude of a signal. As the name suggests, they are calculated by taking the square root of the mean of squares of values. Unlike other spectral features, RMS values are not calculated from the FFT. 

### librosa.feature.spectral_centroid: 

This feature characterizes the “center of mass” of a frequency spectrum. The spectral centroid is computed as the weighted mean of the frequencies present in the sound, where the weights are the magnitudes of the frequencies in the spectrum.  A violin playing high notes will have a higher spectral centroid compared to a bass guitar playing low notes.

### librosa.feature.spectral_contrast: 
Feature that describes the difference in energy levels between high-energy frequencies and low-energy frequencies of a frequency spectrum. Spectral contrast is similar to spectral bandwidth and spectral centroid, however it particularly emphasizes the dynamic range within frequency bands.

### librosa.feature.spectral_bandwith:
Measure of the spread of frequencies in an audio signal around a central frequency, often the spectral centroid. Because it measures spread, calculation of spectral bandwidth is similar to standard deviation.

### librosa.feature.spectral_flatness:
Feature used to describe how "flat" or "peaky" the frequency spectrum of a sound is. A pure sine wave has most of its energy concentrated at a single frequency, so the spectral flatness is low (close to 0). Meanwhile, white noise has equal energy across all frequencies, so the spectral flatness is high (close to 1). Spectral flatness is particularly useful in distinguishing between tonal and noise-like sounds.

### librosa.feature.spectral_rolloff:
Feature used to describe the frequency below which a specified percentage (typically 85% or 95%) of the total spectral energy is concentrated. It is a way of quantifying how the energy in an audio signal is distributed across the frequency spectrum.

### Mel spectrogram: 
I tested models with raw flattened mel spectrogram input. However, this actually decreased the accuracy of the models. For instance, the accuracy of gradient boosting decreased from 70% to 50% when the raw Mel spectrogram was included. The data included in the raw mel spectrogram was too high a dimensionality and too redundant with the other features to be worth it. 

## Model Selection:

I compared three models: k-NN, SVM, and gradient boosting. I found that I could get a 10-20% increase using gradient boosting over k-NN and SVM. Additionally, given that sklearn has an efficient implementation of gradient boosting using histograms it was not much more computationally costly compared to the other models. Finally, I am interested in gradient boosting because it plays a significant role in my undergraduate research in a biomedical engineering lab, where we use it to classify labelled spectroscopy data collected from blood cells.

### Summary of Gradient Boosting:

The basic unit in modern gradient boosting algorithms is a decision tree. A decision tree is a simple model for classification. It resembles a human made decision tree, such as a flowchart. The tree is made via recursive partitioning. Recursive partitioning can be approached through several mathematical strategies, but it essentially just aims to minimize the amount of misclassified data for a given feature. One way to do this is with Gini Impurity, which quantifies how “impure” a partition is. It is one minus the sum of the frequency of each class squared, so 0 is perfectly pure, and an even mix of four classes would be .75, which is impure. A decision tree will take the feature that best splits the data and split the whole data set (or root node) into two parts. Then, the resulting two nodes are split again, continuing recursively to a certain endpoint defined by the hyperparameters. At the end of each node is a “leaf”, or the best classification for the data in the node. 

![A simple decision tree. Source: Wikipedia](https://upload.wikimedia.org/wikipedia/commons/e/eb/Decision_Tree.jpg)
An example of a simple decision tree.

Gradient boosting is an ensemble approach, meaning that it combines many less accurate decision trees into a more powerful model. The algorithm to do so is called “boosting”, hence its name. Boosting in classification is based on making a new tree on the gradient of predicted log-odds, which is a measure of the residuals (errors) of the previous attempt. The update to the new tree is scaled by a hyperparameter called shrinkage. When using gradient boosting algorithms, one tradeoff is scaling this shrinkage, in addition to the number of trees the model iterates through. A high shrinkage allows the model to be trained with fewer iterations; however, it increases the chance of overfitting the data. Meanwhile, the “careful” approach of a low shrinkage and many estimators is more computationally intensive, but produces a more accurate model.

### Hyperparameter Optimization

Tuning shrinkage and the number of estimators was difficult due to the size of the dataset and number of features being inputted into the model. I settled on a shrinkage rate of .01 and a maximum number of estimators of 1500. These hyperparameters increased the accuracy by 2% while still keeping the program’s training time to under an hour on my laptop.

## Conclusion:
The model reached an final accuracy of 78.66%. To further improve model performance, I could improve the feature selection of the model via finding new features outside of librosa. Additionally, the gradient boosting algorithm has more parameters than just shrinkage and number of estimators, these could be tuned systematically as well. Overall, I found this exercise helpful for acquainting myself with data science workflows and hope to apply a modern approach such as ANNs for my next project.

### Code cells used for experimenting with and testing the model follow below.

In [4]:
# Machine Learning Birdcall Classifier
# for an intro to ML workflows and practice with basic feature extraction.
# Owen Karmel

# imports
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import librosa
from scipy.io import wavfile
from tqdm import tqdm
from sklearn import model_selection
from sklearn import preprocessing
from sklearn import neighbors
from sklearn import svm
from sklearn import ensemble
from sklearn.experimental import enable_halving_search_cv

train_x_size = 12000  # maximum in training set: 12000
test_x_size = 3000  # maximum in test set: 3000
wavfile_length = 24000

# Set up training set
t1 = time.time()
df = pd.read_csv('train.csv')  # Read train.csv
train_x = np.empty((train_x_size, wavfile_length))  # initialize train_x
path = 'train/train/'  # path from root directory to training .wav files
print('creating variable train_x')
for i in tqdm(range(train_x_size)):  # Creates train_x from wav files
    train_x[i] = np.array(wavfile.read(path+f'train_{i}.wav')[1])
train_y = df.Label[:train_x_size]  # train y is just the labels column from the csv

print('train_x to np.array time: ', time.time()-t1)  # time to run
sr = wavfile.read(f'train/train/train_{i}.wav')[0] # determines sample rate

# Set up test set
test_x = np.empty((test_x_size, wavfile_length))  # initialize test_x
test_ids = np.empty(test_x_size,dtype='int32')
path = 'train/train/'  # path from root directory to training .wav files
print('creating variable test_x')
for i in tqdm(range(test_x_size)):  # creates test_x from wav files
    test_x[i] = np.array(wavfile.read(path+f'train_{12000-test_x_size+i}.wav')[1])
    test_ids[i] = i

test_y = df.Label[-1*test_x_size:]

print('test_x to np.array time: ', time.time()-t1)  # time to run

ModuleNotFoundError: No module named 'pandas'

In [22]:
# Feature Extraction
def extract_librosa_features(y, sr):
    """
    extract auditory features from wavefiles
    :param y: ndarray of wavfile
    :param sr: sampling rate of wavfile
    :return: flattened string of the features mfcc,
    rms spectral_centroid, spectral_bandwidth, spectral_flatness and
    spectral_rolloff
    """
    return np.concatenate((
        np.ndarray.flatten(librosa.feature.mfcc(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.rms(y=y)),
        np.ndarray.flatten(librosa.feature.spectral_centroid(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.spectral_contrast(y=y,sr=sr,fmin=100)),
        np.ndarray.flatten(librosa.feature.spectral_bandwidth(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.spectral_flatness(y=y)),
        #np.ndarray.flatten(librosa.feature.spectral_rolloff(y=y,sr=sr))

    ))


print('extracting librosa features for train_x:')
train_x_features = np.array([extract_librosa_features(sample, sr)
                             for sample in tqdm(train_x)])
train_x_features = preprocessing.StandardScaler().fit_transform(train_x_features)

print('extracting librosa features for test_x:')
test_x_features = np.array([extract_librosa_features(sample, sr)
                            for sample in tqdm(test_x)])
test_x_features = preprocessing.StandardScaler().fit_transform(test_x_features)

print('#######Gradient Boosting on Librosa Features########')
GBC = ensemble.HistGradientBoostingClassifier(learning_rate=.15, max_iter=40,verbose=True)
start= time.time()
GBC.fit(train_x_features,train_y)
interval1 = time.time()
print(f'GBC model fit time: {interval1-start:.2f} s')
print(f'training accuracy: {GBC.score(train_x_features,train_y)}')
interval2 = time.time()
print(f'train predict time: {interval2-interval1:.2f} s')
print(f'test accuracy: {GBC.score(test_x_features, test_y):.4f}')
interval3 = time.time()
print(f'test predict time: {interval3-interval2:.2f} s')

extracting librosa features for train_x:


100%|███████████| 11000/11000 [11:03<00:00, 16.58it/s]


extracting librosa features for test_x:


100%|█████████████| 1000/1000 [01:00<00:00, 16.55it/s]


#######Gradient Boosting on Librosa Features########
Binning 0.115 GB of training data: 5.047 s
Binning 0.013 GB of validation data: 0.157 s
Fitting gradient boosted rounds:
[1/40] 5 trees, 155 leaves (31 on avg), max depth = 13, train loss: 1.41589, val loss: 1.46229, in 1.884s
[2/40] 5 trees, 155 leaves (31 on avg), max depth = 15, train loss: 1.27247, val loss: 1.35608, in 1.320s
[3/40] 5 trees, 155 leaves (31 on avg), max depth = 14, train loss: 1.15744, val loss: 1.26916, in 1.376s
[4/40] 5 trees, 155 leaves (31 on avg), max depth = 21, train loss: 1.05814, val loss: 1.19786, in 1.326s
[5/40] 5 trees, 155 leaves (31 on avg), max depth = 20, train loss: 0.97253, val loss: 1.13254, in 1.337s
[6/40] 5 trees, 155 leaves (31 on avg), max depth = 18, train loss: 0.90197, val loss: 1.08063, in 1.351s
[7/40] 5 trees, 155 leaves (31 on avg), max depth = 15, train loss: 0.83954, val loss: 1.03723, in 1.428s
[8/40] 5 trees, 155 leaves (31 on avg), max depth = 19, train loss: 0.78506, val los

In [23]:
# Feature Extraction
def extract_librosa_features(y, sr):
    """
    extract auditory features from wavefiles
    :param y: ndarray of wavfile
    :param sr: sampling rate of wavfile
    :return: flattened string of the features mfcc,
    rms spectral_centroid, spectral_bandwidth, spectral_flatness and
    spectral_rolloff
    """
    return np.concatenate((
        np.ndarray.flatten(librosa.feature.mfcc(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.rms(y=y)),
        np.ndarray.flatten(librosa.feature.spectral_centroid(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.spectral_contrast(y=y,sr=sr,fmin=100)),
        np.ndarray.flatten(librosa.feature.spectral_bandwidth(y=y,sr=sr)),
        #np.ndarray.flatten(librosa.feature.spectral_flatness(y=y)),
        np.ndarray.flatten(librosa.feature.spectral_rolloff(y=y,sr=sr))

    ))


print('extracting librosa features for train_x:')
train_x_features = np.array([extract_librosa_features(sample, sr)
                             for sample in tqdm(train_x)])
train_x_features = preprocessing.StandardScaler().fit_transform(train_x_features)

print('extracting librosa features for test_x:')
test_x_features = np.array([extract_librosa_features(sample, sr)
                            for sample in tqdm(test_x)])
test_x_features = preprocessing.StandardScaler().fit_transform(test_x_features)

print('#######Gradient Boosting on Librosa Features########')
GBC = ensemble.HistGradientBoostingClassifier(learning_rate=.15, max_iter=40,verbose=True)
start= time.time()
GBC.fit(train_x_features,train_y)
interval1 = time.time()
print(f'GBC model fit time: {interval1-start:.2f} s')
print(f'training accuracy: {GBC.score(train_x_features,train_y)}')
interval2 = time.time()
print(f'train predict time: {interval2-interval1:.2f} s')
print(f'test accuracy: {GBC.score(test_x_features, test_y):.4f}')
interval3 = time.time()
print(f'test predict time: {interval3-interval2:.2f} s')

extracting librosa features for train_x:


100%|███████████| 11000/11000 [12:48<00:00, 14.32it/s]


extracting librosa features for test_x:


100%|█████████████| 1000/1000 [01:09<00:00, 14.31it/s]


#######Gradient Boosting on Librosa Features########
Binning 0.115 GB of training data: 4.087 s
Binning 0.013 GB of validation data: 0.353 s
Fitting gradient boosted rounds:
[1/40] 5 trees, 155 leaves (31 on avg), max depth = 12, train loss: 1.40528, val loss: 1.46589, in 4.004s
[2/40] 5 trees, 155 leaves (31 on avg), max depth = 15, train loss: 1.26063, val loss: 1.35789, in 3.456s
[3/40] 5 trees, 155 leaves (31 on avg), max depth = 20, train loss: 1.14386, val loss: 1.27594, in 2.338s
[4/40] 5 trees, 155 leaves (31 on avg), max depth = 14, train loss: 1.04284, val loss: 1.20345, in 3.172s
[5/40] 5 trees, 155 leaves (31 on avg), max depth = 19, train loss: 0.96089, val loss: 1.14676, in 5.715s
[6/40] 5 trees, 155 leaves (31 on avg), max depth = 21, train loss: 0.89152, val loss: 1.10043, in 3.764s
[7/40] 5 trees, 155 leaves (31 on avg), max depth = 14, train loss: 0.82941, val loss: 1.06058, in 5.270s
[8/40] 5 trees, 155 leaves (31 on avg), max depth = 17, train loss: 0.77613, val los

In [24]:
# Feature Extraction
def extract_librosa_features(y, sr):
    """
    extract auditory features from wavefiles
    :param y: ndarray of wavfile
    :param sr: sampling rate of wavfile
    :return: flattened string of the features mfcc,
    rms spectral_centroid, spectral_bandwidth, spectral_flatness and
    spectral_rolloff
    """
    return np.concatenate((
        np.ndarray.flatten(librosa.feature.mfcc(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.rms(y=y)),
        np.ndarray.flatten(librosa.feature.spectral_centroid(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.spectral_contrast(y=y,sr=sr,fmin=100)),
        #np.ndarray.flatten(librosa.feature.spectral_bandwidth(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.spectral_flatness(y=y)),
        np.ndarray.flatten(librosa.feature.spectral_rolloff(y=y,sr=sr))

    ))


print('extracting librosa features for train_x:')
train_x_features = np.array([extract_librosa_features(sample, sr)
                             for sample in tqdm(train_x)])
train_x_features = preprocessing.StandardScaler().fit_transform(train_x_features)

print('extracting librosa features for test_x:')
test_x_features = np.array([extract_librosa_features(sample, sr)
                            for sample in tqdm(test_x)])
test_x_features = preprocessing.StandardScaler().fit_transform(test_x_features)

print('#######Gradient Boosting on Librosa Features########')
GBC = ensemble.HistGradientBoostingClassifier(learning_rate=.15, max_iter=40,verbose=True)
start= time.time()
GBC.fit(train_x_features,train_y)
interval1 = time.time()
print(f'GBC model fit time: {interval1-start:.2f} s')
print(f'training accuracy: {GBC.score(train_x_features,train_y)}')
interval2 = time.time()
print(f'train predict time: {interval2-interval1:.2f} s')
print(f'test accuracy: {GBC.score(test_x_features, test_y):.4f}')
interval3 = time.time()
print(f'test predict time: {interval3-interval2:.2f} s')

extracting librosa features for train_x:


100%|███████████| 11000/11000 [16:01<00:00, 11.44it/s]


extracting librosa features for test_x:


100%|█████████████| 1000/1000 [01:16<00:00, 13.07it/s]


#######Gradient Boosting on Librosa Features########
Binning 0.115 GB of training data: 9.023 s
Binning 0.013 GB of validation data: 0.997 s
Fitting gradient boosted rounds:
[1/40] 5 trees, 155 leaves (31 on avg), max depth = 13, train loss: 1.40654, val loss: 1.45805, in 6.209s
[2/40] 5 trees, 155 leaves (31 on avg), max depth = 13, train loss: 1.26103, val loss: 1.35980, in 5.306s
[3/40] 5 trees, 155 leaves (31 on avg), max depth = 14, train loss: 1.14472, val loss: 1.27384, in 5.890s
[4/40] 5 trees, 155 leaves (31 on avg), max depth = 19, train loss: 1.04734, val loss: 1.20812, in 5.673s
[5/40] 5 trees, 155 leaves (31 on avg), max depth = 16, train loss: 0.96343, val loss: 1.14745, in 6.078s
[6/40] 5 trees, 155 leaves (31 on avg), max depth = 21, train loss: 0.89283, val loss: 1.10548, in 4.976s
[7/40] 5 trees, 155 leaves (31 on avg), max depth = 17, train loss: 0.83240, val loss: 1.06877, in 5.055s
[8/40] 5 trees, 155 leaves (31 on avg), max depth = 19, train loss: 0.77865, val los

In [25]:
# Feature Extraction
def extract_librosa_features(y, sr):
    """
    extract auditory features from wavefiles
    :param y: ndarray of wavfile
    :param sr: sampling rate of wavfile
    :return: flattened string of the features mfcc,
    rms spectral_centroid, spectral_bandwidth, spectral_flatness and
    spectral_rolloff
    """
    return np.concatenate((
        np.ndarray.flatten(librosa.feature.mfcc(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.rms(y=y)),
        np.ndarray.flatten(librosa.feature.spectral_centroid(y=y,sr=sr)),
        #np.ndarray.flatten(librosa.feature.spectral_contrast(y=y,sr=sr,fmin=100)),
        np.ndarray.flatten(librosa.feature.spectral_bandwidth(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.spectral_flatness(y=y)),
        np.ndarray.flatten(librosa.feature.spectral_rolloff(y=y,sr=sr))

    ))


print('extracting librosa features for train_x:')
train_x_features = np.array([extract_librosa_features(sample, sr)
                             for sample in tqdm(train_x)])
train_x_features = preprocessing.StandardScaler().fit_transform(train_x_features)

print('extracting librosa features for test_x:')
test_x_features = np.array([extract_librosa_features(sample, sr)
                            for sample in tqdm(test_x)])
test_x_features = preprocessing.StandardScaler().fit_transform(test_x_features)

print('#######Gradient Boosting on Librosa Features########')
GBC = ensemble.HistGradientBoostingClassifier(learning_rate=.15, max_iter=40,verbose=True)
start= time.time()
GBC.fit(train_x_features,train_y)
interval1 = time.time()
print(f'GBC model fit time: {interval1-start:.2f} s')
print(f'training accuracy: {GBC.score(train_x_features,train_y)}')
interval2 = time.time()
print(f'train predict time: {interval2-interval1:.2f} s')
print(f'test accuracy: {GBC.score(test_x_features, test_y):.4f}')
interval3 = time.time()
print(f'test predict time: {interval3-interval2:.2f} s')

extracting librosa features for train_x:


100%|███████████| 11000/11000 [09:40<00:00, 18.95it/s]


extracting librosa features for test_x:


100%|█████████████| 1000/1000 [00:26<00:00, 38.13it/s]


#######Gradient Boosting on Librosa Features########
Binning 0.093 GB of training data: 3.030 s
Binning 0.010 GB of validation data: 0.164 s
Fitting gradient boosted rounds:
[1/40] 5 trees, 155 leaves (31 on avg), max depth = 12, train loss: 1.41236, val loss: 1.45127, in 2.403s
[2/40] 5 trees, 155 leaves (31 on avg), max depth = 11, train loss: 1.26779, val loss: 1.34589, in 2.217s
[3/40] 5 trees, 155 leaves (31 on avg), max depth = 16, train loss: 1.15451, val loss: 1.26106, in 2.271s
[4/40] 5 trees, 155 leaves (31 on avg), max depth = 20, train loss: 1.05638, val loss: 1.18523, in 2.302s
[5/40] 5 trees, 155 leaves (31 on avg), max depth = 18, train loss: 0.97509, val loss: 1.12770, in 3.471s
[6/40] 5 trees, 155 leaves (31 on avg), max depth = 18, train loss: 0.90166, val loss: 1.07670, in 3.900s
[7/40] 5 trees, 155 leaves (31 on avg), max depth = 17, train loss: 0.83949, val loss: 1.03577, in 2.609s
[8/40] 5 trees, 155 leaves (31 on avg), max depth = 15, train loss: 0.78351, val los

In [26]:
# Feature Extraction
def extract_librosa_features(y, sr):
    """
    extract auditory features from wavefiles
    :param y: ndarray of wavfile
    :param sr: sampling rate of wavfile
    :return: flattened string of the features mfcc,
    rms spectral_centroid, spectral_bandwidth, spectral_flatness and
    spectral_rolloff
    """
    return np.concatenate((
        np.ndarray.flatten(librosa.feature.mfcc(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.rms(y=y)),
        #np.ndarray.flatten(librosa.feature.spectral_centroid(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.spectral_contrast(y=y,sr=sr,fmin=100)),
        np.ndarray.flatten(librosa.feature.spectral_bandwidth(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.spectral_flatness(y=y)),
        np.ndarray.flatten(librosa.feature.spectral_rolloff(y=y,sr=sr))

    ))


print('extracting librosa features for train_x:')
train_x_features = np.array([extract_librosa_features(sample, sr)
                             for sample in tqdm(train_x)])
train_x_features = preprocessing.StandardScaler().fit_transform(train_x_features)

print('extracting librosa features for test_x:')
test_x_features = np.array([extract_librosa_features(sample, sr)
                            for sample in tqdm(test_x)])
test_x_features = preprocessing.StandardScaler().fit_transform(test_x_features)

print('#######Gradient Boosting on Librosa Features########')
GBC = ensemble.HistGradientBoostingClassifier(learning_rate=.15, max_iter=40,verbose=True)
start= time.time()
GBC.fit(train_x_features,train_y)
interval1 = time.time()
print(f'GBC model fit time: {interval1-start:.2f} s')
print(f'training accuracy: {GBC.score(train_x_features,train_y)}')
interval2 = time.time()
print(f'train predict time: {interval2-interval1:.2f} s')
print(f'test accuracy: {GBC.score(test_x_features, test_y):.4f}')
interval3 = time.time()
print(f'test predict time: {interval3-interval2:.2f} s')

extracting librosa features for train_x:


100%|███████████| 11000/11000 [09:29<00:00, 19.31it/s]


extracting librosa features for test_x:


100%|█████████████| 1000/1000 [01:21<00:00, 12.32it/s]


#######Gradient Boosting on Librosa Features########
Binning 0.115 GB of training data: 7.234 s
Binning 0.013 GB of validation data: 0.547 s
Fitting gradient boosted rounds:
[1/40] 5 trees, 155 leaves (31 on avg), max depth = 13, train loss: 1.40742, val loss: 1.45413, in 5.035s
[2/40] 5 trees, 155 leaves (31 on avg), max depth = 13, train loss: 1.26337, val loss: 1.34007, in 4.731s
[3/40] 5 trees, 155 leaves (31 on avg), max depth = 15, train loss: 1.14674, val loss: 1.25055, in 5.009s
[4/40] 5 trees, 155 leaves (31 on avg), max depth = 16, train loss: 1.05031, val loss: 1.17742, in 4.663s
[5/40] 5 trees, 155 leaves (31 on avg), max depth = 19, train loss: 0.96893, val loss: 1.11729, in 4.262s
[6/40] 5 trees, 155 leaves (31 on avg), max depth = 20, train loss: 0.89862, val loss: 1.06894, in 3.684s
[7/40] 5 trees, 155 leaves (31 on avg), max depth = 16, train loss: 0.83638, val loss: 1.02389, in 4.018s
[8/40] 5 trees, 155 leaves (31 on avg), max depth = 20, train loss: 0.78049, val los

In [27]:
# Feature Extraction
def extract_librosa_features(y, sr):
    """
    extract auditory features from wavefiles
    :param y: ndarray of wavfile
    :param sr: sampling rate of wavfile
    :return: flattened string of the features mfcc,
    rms spectral_centroid, spectral_bandwidth, spectral_flatness and
    spectral_rolloff
    """
    return np.concatenate((
        np.ndarray.flatten(librosa.feature.mfcc(y=y,sr=sr)),
        #np.ndarray.flatten(librosa.feature.rms(y=y)),
        np.ndarray.flatten(librosa.feature.spectral_centroid(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.spectral_contrast(y=y,sr=sr,fmin=100)),
        np.ndarray.flatten(librosa.feature.spectral_bandwidth(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.spectral_flatness(y=y)),
        np.ndarray.flatten(librosa.feature.spectral_rolloff(y=y,sr=sr))

    ))


print('extracting librosa features for train_x:')
train_x_features = np.array([extract_librosa_features(sample, sr)
                             for sample in tqdm(train_x)])
train_x_features = preprocessing.StandardScaler().fit_transform(train_x_features)

print('extracting librosa features for test_x:')
test_x_features = np.array([extract_librosa_features(sample, sr)
                            for sample in tqdm(test_x)])
test_x_features = preprocessing.StandardScaler().fit_transform(test_x_features)

print('#######Gradient Boosting on Librosa Features########')
GBC = ensemble.HistGradientBoostingClassifier(learning_rate=.15, max_iter=40,verbose=True)
start= time.time()
GBC.fit(train_x_features,train_y)
interval1 = time.time()
print(f'GBC model fit time: {interval1-start:.2f} s')
print(f'training accuracy: {GBC.score(train_x_features,train_y)}')
interval2 = time.time()
print(f'train predict time: {interval2-interval1:.2f} s')
print(f'test accuracy: {GBC.score(test_x_features, test_y):.4f}')
interval3 = time.time()
print(f'test predict time: {interval3-interval2:.2f} s')

extracting librosa features for train_x:


100%|███████████| 11000/11000 [11:23<00:00, 16.08it/s]


extracting librosa features for test_x:


100%|█████████████| 1000/1000 [00:51<00:00, 19.58it/s]

#######Gradient Boosting on Librosa Features########
Binning 0.115 GB of training data: 

7.624 s
Binning 0.013 GB of validation data: 0.777 s
Fitting gradient boosted rounds:
[1/40] 5 trees, 155 leaves (31 on avg), max depth = 14, train loss: 1.40709, val loss: 1.46043, in 7.406s
[2/40] 5 trees, 155 leaves (31 on avg), max depth = 16, train loss: 1.26137, val loss: 1.35199, in 5.699s
[3/40] 5 trees, 155 leaves (31 on avg), max depth = 15, train loss: 1.14444, val loss: 1.27199, in 3.932s
[4/40] 5 trees, 155 leaves (31 on avg), max depth = 15, train loss: 1.04862, val loss: 1.19772, in 3.861s
[5/40] 5 trees, 155 leaves (31 on avg), max depth = 17, train loss: 0.96609, val loss: 1.14367, in 3.383s
[6/40] 5 trees, 155 leaves (31 on avg), max depth = 18, train loss: 0.89493, val loss: 1.08715, in 3.800s
[7/40] 5 trees, 155 leaves (31 on avg), max depth = 15, train loss: 0.83172, val loss: 1.04318, in 2.917s
[8/40] 5 trees, 155 leaves (31 on avg), max depth = 18, train loss: 0.77703, val loss: 1.00567, in 3.771s
[9/40] 5 trees, 155 leaves (31 on avg), max depth = 19, train loss

In [28]:
# Feature Extraction
def extract_librosa_features(y, sr):
    """
    extract auditory features from wavefiles
    :param y: ndarray of wavfile
    :param sr: sampling rate of wavfile
    :return: flattened string of the features mfcc,
    rms spectral_centroid, spectral_bandwidth, spectral_flatness and
    spectral_rolloff
    """
    return np.concatenate((
        #np.ndarray.flatten(librosa.feature.mfcc(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.rms(y=y)),
        np.ndarray.flatten(librosa.feature.spectral_centroid(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.spectral_contrast(y=y,sr=sr,fmin=100)),
        np.ndarray.flatten(librosa.feature.spectral_bandwidth(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.spectral_flatness(y=y)),
        np.ndarray.flatten(librosa.feature.spectral_rolloff(y=y,sr=sr))

    ))


print('extracting librosa features for train_x:')
train_x_features = np.array([extract_librosa_features(sample, sr)
                             for sample in tqdm(train_x)])
train_x_features = preprocessing.StandardScaler().fit_transform(train_x_features)

print('extracting librosa features for test_x:')
test_x_features = np.array([extract_librosa_features(sample, sr)
                            for sample in tqdm(test_x)])
test_x_features = preprocessing.StandardScaler().fit_transform(test_x_features)

print('#######Gradient Boosting on Librosa Features########')
GBC = ensemble.HistGradientBoostingClassifier(learning_rate=.15, max_iter=40,verbose=True)
start= time.time()
GBC.fit(train_x_features,train_y)
interval1 = time.time()
print(f'GBC model fit time: {interval1-start:.2f} s')
print(f'training accuracy: {GBC.score(train_x_features,train_y)}')
interval2 = time.time()
print(f'train predict time: {interval2-interval1:.2f} s')
print(f'test accuracy: {GBC.score(test_x_features, test_y):.4f}')
interval3 = time.time()
print(f'test predict time: {interval3-interval2:.2f} s')

extracting librosa features for train_x:


100%|███████████| 11000/11000 [05:02<00:00, 36.35it/s]


extracting librosa features for test_x:


100%|█████████████| 1000/1000 [00:24<00:00, 40.45it/s]

#######Gradient Boosting on Librosa Features########
Binning 0.045 GB of training data: 

1.462 s
Binning 0.005 GB of validation data: 0.156 s
Fitting gradient boosted rounds:
[1/40] 5 trees, 155 leaves (31 on avg), max depth = 13, train loss: 1.43104, val loss: 1.47120, in 1.615s
[2/40] 5 trees, 155 leaves (31 on avg), max depth = 18, train loss: 1.29707, val loss: 1.36336, in 1.316s
[3/40] 5 trees, 155 leaves (31 on avg), max depth = 15, train loss: 1.19337, val loss: 1.28663, in 1.386s
[4/40] 5 trees, 155 leaves (31 on avg), max depth = 21, train loss: 1.10650, val loss: 1.21642, in 1.608s
[5/40] 5 trees, 155 leaves (31 on avg), max depth = 14, train loss: 1.03615, val loss: 1.16789, in 1.392s
[6/40] 5 trees, 155 leaves (31 on avg), max depth = 20, train loss: 0.97192, val loss: 1.12574, in 1.497s
[7/40] 5 trees, 155 leaves (31 on avg), max depth = 19, train loss: 0.91769, val loss: 1.08932, in 1.575s
[8/40] 5 trees, 155 leaves (31 on avg), max depth = 16, train loss: 0.86752, val loss: 1.05372, in 1.485s
[9/40] 5 trees, 155 leaves (31 on avg), max depth = 17, train loss

In [27]:
# Feature Extraction
def extract_librosa_features(y, sr):
    """
    extract auditory features from wavefiles
    :param y: ndarray of wavfile
    :param sr: sampling rate of wavfile
    :return: flattened string of the features mfcc,
    rms spectral_centroid, spectral_bandwidth, spectral_flatness and
    spectral_rolloff
    """
    return np.concatenate((
        np.ndarray.flatten(librosa.feature.mfcc(y=y,sr=sr)),
        #np.ndarray.flatten(librosa.feature.rms(y=y)),
        np.ndarray.flatten(librosa.feature.spectral_centroid(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.spectral_contrast(y=y,sr=sr,fmin=100)),
        np.ndarray.flatten(librosa.feature.spectral_bandwidth(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.spectral_flatness(y=y)),
        np.ndarray.flatten(librosa.feature.spectral_rolloff(y=y,sr=sr))

    ))


print('extracting librosa features for train_x:')
train_x_features = np.array([extract_librosa_features(sample, sr)
                             for sample in tqdm(train_x)])
train_x_features = preprocessing.StandardScaler().fit_transform(train_x_features)

print('extracting librosa features for test_x:')
test_x_features = np.array([extract_librosa_features(sample, sr)
                            for sample in tqdm(test_x)])
test_x_features = preprocessing.StandardScaler().fit_transform(test_x_features)

print('#######Gradient Boosting on Librosa Features########')
GBC = ensemble.HistGradientBoostingClassifier(learning_rate=.15, max_iter=40,verbose=True)
start= time.time()
GBC.fit(train_x_features,train_y)
interval1 = time.time()
print(f'GBC model fit time: {interval1-start:.2f} s')
print(f'training accuracy: {GBC.score(train_x_features,train_y)}')
interval2 = time.time()
print(f'train predict time: {interval2-interval1:.2f} s')
print(f'test accuracy: {GBC.score(test_x_features, test_y):.4f}')
interval3 = time.time()
print(f'test predict time: {interval3-interval2:.2f} s')

extracting librosa features for train_x:


100%|███████████| 11000/11000 [11:23<00:00, 16.08it/s]


extracting librosa features for test_x:


100%|█████████████| 1000/1000 [00:51<00:00, 19.58it/s]

#######Gradient Boosting on Librosa Features########
Binning 0.115 GB of training data: 

7.624 s
Binning 0.013 GB of validation data: 0.777 s
Fitting gradient boosted rounds:
[1/40] 5 trees, 155 leaves (31 on avg), max depth = 14, train loss: 1.40709, val loss: 1.46043, in 7.406s
[2/40] 5 trees, 155 leaves (31 on avg), max depth = 16, train loss: 1.26137, val loss: 1.35199, in 5.699s
[3/40] 5 trees, 155 leaves (31 on avg), max depth = 15, train loss: 1.14444, val loss: 1.27199, in 3.932s
[4/40] 5 trees, 155 leaves (31 on avg), max depth = 15, train loss: 1.04862, val loss: 1.19772, in 3.861s
[5/40] 5 trees, 155 leaves (31 on avg), max depth = 17, train loss: 0.96609, val loss: 1.14367, in 3.383s
[6/40] 5 trees, 155 leaves (31 on avg), max depth = 18, train loss: 0.89493, val loss: 1.08715, in 3.800s
[7/40] 5 trees, 155 leaves (31 on avg), max depth = 15, train loss: 0.83172, val loss: 1.04318, in 2.917s
[8/40] 5 trees, 155 leaves (31 on avg), max depth = 18, train loss: 0.77703, val loss: 1.00567, in 3.771s
[9/40] 5 trees, 155 leaves (31 on avg), max depth = 19, train loss

In [36]:
# Feature Extraction
def extract_librosa_features(y, sr):
    """
    extract auditory features from wavefiles
    :param y: ndarray of wavfile
    :param sr: sampling rate of wavfile
    :return: flattened string of the features mfcc,
    rms spectral_centroid, spectral_bandwidth, spectral_flatness and
    spectral_rolloff
    """
    return np.concatenate((
        np.ndarray.flatten(librosa.feature.mfcc(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.rms(y=y)),
        #np.ndarray.flatten(librosa.feature.spectral_centroid(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.spectral_contrast(y=y,sr=sr,fmin=100)),
        np.ndarray.flatten(librosa.feature.spectral_bandwidth(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.spectral_flatness(y=y)),
        #np.ndarray.flatten(librosa.feature.spectral_rolloff(y=y,sr=sr))

    ))


print('extracting librosa features for train_x:')
train_x_features = np.array([extract_librosa_features(sample, sr)
                             for sample in tqdm(train_x)])
train_x_features = preprocessing.StandardScaler().fit_transform(train_x_features)

print('extracting librosa features for test_x:')
test_x_features = np.array([extract_librosa_features(sample, sr)
                            for sample in tqdm(test_x)])
test_x_features = preprocessing.StandardScaler().fit_transform(test_x_features)

print('#######Gradient Boosting on Librosa Features########')
GBC = ensemble.HistGradientBoostingClassifier(learning_rate=.15, max_iter=40,verbose=True)
start= time.time()
GBC.fit(train_x_features,train_y)
interval1 = time.time()
print(f'GBC model fit time: {interval1-start:.2f} s')
print(f'training accuracy: {GBC.score(train_x_features,train_y)}')
interval2 = time.time()
print(f'train predict time: {interval2-interval1:.2f} s')
print(f'test accuracy: {GBC.score(test_x_features, test_y):.4f}')
interval3 = time.time()
print(f'test predict time: {interval3-interval2:.2f} s')

extracting librosa features for train_x:


100%|███████████| 11000/11000 [13:39<00:00, 13.42it/s]


extracting librosa features for test_x:


100%|█████████████| 1000/1000 [00:46<00:00, 21.44it/s]


#######Gradient Boosting on Librosa Features########
Binning 0.112 GB of training data: 4.109 s
Binning 0.012 GB of validation data: 0.150 s
Fitting gradient boosted rounds:
[1/40] 5 trees, 155 leaves (31 on avg), max depth = 14, train loss: 1.40724, val loss: 1.47171, in 1.256s
[2/40] 5 trees, 155 leaves (31 on avg), max depth = 14, train loss: 1.26608, val loss: 1.36741, in 1.211s
[3/40] 5 trees, 155 leaves (31 on avg), max depth = 20, train loss: 1.15062, val loss: 1.28583, in 1.193s
[4/40] 5 trees, 155 leaves (31 on avg), max depth = 20, train loss: 1.05178, val loss: 1.21426, in 1.191s
[5/40] 5 trees, 155 leaves (31 on avg), max depth = 18, train loss: 0.97154, val loss: 1.15875, in 1.223s
[6/40] 5 trees, 155 leaves (31 on avg), max depth = 20, train loss: 0.90146, val loss: 1.11708, in 1.209s
[7/40] 5 trees, 155 leaves (31 on avg), max depth = 16, train loss: 0.83997, val loss: 1.07213, in 1.193s
[8/40] 5 trees, 155 leaves (31 on avg), max depth = 15, train loss: 0.78668, val los

In [37]:
# Feature Extraction
def extract_librosa_features(y, sr):
    """
    extract auditory features from wavefiles
    :param y: ndarray of wavfile
    :param sr: sampling rate of wavfile
    :return: flattened string of the features mfcc,
    rms spectral_centroid, spectral_bandwidth, spectral_flatness and
    spectral_rolloff
    """
    return np.concatenate((
        np.ndarray.flatten(librosa.feature.mfcc(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.rms(y=y)),
        #np.ndarray.flatten(librosa.feature.spectral_centroid(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.spectral_contrast(y=y,sr=sr,fmin=100)),
        np.ndarray.flatten(librosa.feature.spectral_bandwidth(y=y,sr=sr)),
        #np.ndarray.flatten(librosa.feature.spectral_flatness(y=y)),
        np.ndarray.flatten(librosa.feature.spectral_rolloff(y=y,sr=sr))

    ))


print('extracting librosa features for train_x:')
train_x_features = np.array([extract_librosa_features(sample, sr)
                             for sample in tqdm(train_x)])
train_x_features = preprocessing.StandardScaler().fit_transform(train_x_features)

print('extracting librosa features for test_x:')
test_x_features = np.array([extract_librosa_features(sample, sr)
                            for sample in tqdm(test_x)])
test_x_features = preprocessing.StandardScaler().fit_transform(test_x_features)

print('#######Gradient Boosting on Librosa Features########')
GBC = ensemble.HistGradientBoostingClassifier(learning_rate=.15, max_iter=40,verbose=True)
start= time.time()
GBC.fit(train_x_features,train_y)
interval1 = time.time()
print(f'GBC model fit time: {interval1-start:.2f} s')
print(f'training accuracy: {GBC.score(train_x_features,train_y)}')
interval2 = time.time()
print(f'train predict time: {interval2-interval1:.2f} s')
print(f'test accuracy: {GBC.score(test_x_features, test_y):.4f}')
interval3 = time.time()
print(f'test predict time: {interval3-interval2:.2f} s')

extracting librosa features for train_x:


100%|█████████| 11000/11000 [9:21:23<00:00,  3.06s/it]


extracting librosa features for test_x:


100%|█████████████| 1000/1000 [00:55<00:00, 18.05it/s]


#######Gradient Boosting on Librosa Features########
Binning 0.112 GB of training data: 4.174 s
Binning 0.012 GB of validation data: 0.158 s
Fitting gradient boosted rounds:
[1/40] 5 trees, 155 leaves (31 on avg), max depth = 12, train loss: 1.40879, val loss: 1.46175, in 1.508s
[2/40] 5 trees, 155 leaves (31 on avg), max depth = 16, train loss: 1.26884, val loss: 1.36698, in 1.447s
[3/40] 5 trees, 155 leaves (31 on avg), max depth = 15, train loss: 1.15096, val loss: 1.28221, in 1.481s
[4/40] 5 trees, 155 leaves (31 on avg), max depth = 15, train loss: 1.05455, val loss: 1.21058, in 1.416s
[5/40] 5 trees, 155 leaves (31 on avg), max depth = 15, train loss: 0.97230, val loss: 1.15053, in 1.416s
[6/40] 5 trees, 155 leaves (31 on avg), max depth = 19, train loss: 0.90126, val loss: 1.10223, in 1.456s
[7/40] 5 trees, 155 leaves (31 on avg), max depth = 17, train loss: 0.84035, val loss: 1.06287, in 1.479s
[8/40] 5 trees, 155 leaves (31 on avg), max depth = 22, train loss: 0.78544, val los

In [38]:
# Feature Extraction
def extract_librosa_features(y, sr):
    """
    extract auditory features from wavefiles
    :param y: ndarray of wavfile
    :param sr: sampling rate of wavfile
    :return: flattened string of the features mfcc,
    rms spectral_centroid, spectral_bandwidth, spectral_flatness and
    spectral_rolloff
    """
    return np.concatenate((
        np.ndarray.flatten(librosa.feature.mfcc(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.rms(y=y)),
        #np.ndarray.flatten(librosa.feature.spectral_centroid(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.spectral_contrast(y=y,sr=sr,fmin=100)),
        #np.ndarray.flatten(librosa.feature.spectral_bandwidth(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.spectral_flatness(y=y)),
        np.ndarray.flatten(librosa.feature.spectral_rolloff(y=y,sr=sr))

    ))


print('extracting librosa features for train_x:')
train_x_features = np.array([extract_librosa_features(sample, sr)
                             for sample in tqdm(train_x)])
train_x_features = preprocessing.StandardScaler().fit_transform(train_x_features)

print('extracting librosa features for test_x:')
test_x_features = np.array([extract_librosa_features(sample, sr)
                            for sample in tqdm(test_x)])
test_x_features = preprocessing.StandardScaler().fit_transform(test_x_features)

print('#######Gradient Boosting on Librosa Features########')
GBC = ensemble.HistGradientBoostingClassifier(learning_rate=.15, max_iter=40,verbose=True)
start= time.time()
GBC.fit(train_x_features,train_y)
interval1 = time.time()
print(f'GBC model fit time: {interval1-start:.2f} s')
print(f'training accuracy: {GBC.score(train_x_features,train_y)}')
interval2 = time.time()
print(f'train predict time: {interval2-interval1:.2f} s')
print(f'test accuracy: {GBC.score(test_x_features, test_y):.4f}')
interval3 = time.time()
print(f'test predict time: {interval3-interval2:.2f} s')

extracting librosa features for train_x:


100%|███████████| 11000/11000 [09:32<00:00, 19.20it/s]


extracting librosa features for test_x:


100%|███████████| 1000/1000 [2:39:34<00:00,  9.57s/it]


#######Gradient Boosting on Librosa Features########
Binning 0.112 GB of training data: 5.289 s
Binning 0.012 GB of validation data: 0.177 s
Fitting gradient boosted rounds:
[1/40] 5 trees, 155 leaves (31 on avg), max depth = 12, train loss: 1.41038, val loss: 1.45754, in 1.863s
[2/40] 5 trees, 155 leaves (31 on avg), max depth = 13, train loss: 1.26502, val loss: 1.34817, in 1.510s
[3/40] 5 trees, 155 leaves (31 on avg), max depth = 14, train loss: 1.15048, val loss: 1.26462, in 1.510s
[4/40] 5 trees, 155 leaves (31 on avg), max depth = 12, train loss: 1.05460, val loss: 1.19354, in 1.567s
[5/40] 5 trees, 155 leaves (31 on avg), max depth = 15, train loss: 0.96871, val loss: 1.12920, in 1.658s
[6/40] 5 trees, 155 leaves (31 on avg), max depth = 13, train loss: 0.89747, val loss: 1.07975, in 1.587s
[7/40] 5 trees, 155 leaves (31 on avg), max depth = 20, train loss: 0.83659, val loss: 1.04195, in 1.522s
[8/40] 5 trees, 155 leaves (31 on avg), max depth = 17, train loss: 0.78078, val los

In [39]:
# Feature Extraction
def extract_librosa_features(y, sr):
    """
    extract auditory features from wavefiles
    :param y: ndarray of wavfile
    :param sr: sampling rate of wavfile
    :return: flattened string of the features mfcc,
    rms spectral_centroid, spectral_bandwidth, spectral_flatness and
    spectral_rolloff
    """
    return np.concatenate((
        np.ndarray.flatten(librosa.feature.mfcc(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.rms(y=y)),
        #np.ndarray.flatten(librosa.feature.spectral_centroid(y=y,sr=sr)),
        #np.ndarray.flatten(librosa.feature.spectral_contrast(y=y,sr=sr,fmin=100)),
        np.ndarray.flatten(librosa.feature.spectral_bandwidth(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.spectral_flatness(y=y)),
        np.ndarray.flatten(librosa.feature.spectral_rolloff(y=y,sr=sr))

    ))


print('extracting librosa features for train_x:')
train_x_features = np.array([extract_librosa_features(sample, sr)
                             for sample in tqdm(train_x)])
train_x_features = preprocessing.StandardScaler().fit_transform(train_x_features)

print('extracting librosa features for test_x:')
test_x_features = np.array([extract_librosa_features(sample, sr)
                            for sample in tqdm(test_x)])
test_x_features = preprocessing.StandardScaler().fit_transform(test_x_features)

print('#######Gradient Boosting on Librosa Features########')
GBC = ensemble.HistGradientBoostingClassifier(learning_rate=.15, max_iter=40,verbose=True)
start= time.time()
GBC.fit(train_x_features,train_y)
interval1 = time.time()
print(f'GBC model fit time: {interval1-start:.2f} s')
print(f'training accuracy: {GBC.score(train_x_features,train_y)}')
interval2 = time.time()
print(f'train predict time: {interval2-interval1:.2f} s')
print(f'test accuracy: {GBC.score(test_x_features, test_y):.4f}')
interval3 = time.time()
print(f'test predict time: {interval3-interval2:.2f} s')

extracting librosa features for train_x:


100%|███████████| 11000/11000 [11:04<00:00, 16.55it/s]


extracting librosa features for test_x:


100%|█████████████| 1000/1000 [01:04<00:00, 15.59it/s]


#######Gradient Boosting on Librosa Features########
Binning 0.089 GB of training data: 3.409 s
Binning 0.010 GB of validation data: 0.174 s
Fitting gradient boosted rounds:
[1/40] 5 trees, 155 leaves (31 on avg), max depth = 12, train loss: 1.41223, val loss: 1.45831, in 2.810s
[2/40] 5 trees, 155 leaves (31 on avg), max depth = 14, train loss: 1.26698, val loss: 1.35154, in 3.024s
[3/40] 5 trees, 155 leaves (31 on avg), max depth = 18, train loss: 1.15033, val loss: 1.26559, in 2.366s
[4/40] 5 trees, 155 leaves (31 on avg), max depth = 19, train loss: 1.05366, val loss: 1.19684, in 2.272s
[5/40] 5 trees, 155 leaves (31 on avg), max depth = 18, train loss: 0.97118, val loss: 1.13995, in 2.314s
[6/40] 5 trees, 155 leaves (31 on avg), max depth = 21, train loss: 0.90088, val loss: 1.08603, in 2.145s
[7/40] 5 trees, 155 leaves (31 on avg), max depth = 17, train loss: 0.83992, val loss: 1.04178, in 2.587s
[8/40] 5 trees, 155 leaves (31 on avg), max depth = 20, train loss: 0.78603, val los

In [40]:
# Feature Extraction
def extract_librosa_features(y, sr):
    """
    extract auditory features from wavefiles
    :param y: ndarray of wavfile
    :param sr: sampling rate of wavfile
    :return: flattened string of the features mfcc,
    rms spectral_centroid, spectral_bandwidth, spectral_flatness and
    spectral_rolloff
    """
    return np.concatenate((
        np.ndarray.flatten(librosa.feature.mfcc(y=y,sr=sr)),
        #np.ndarray.flatten(librosa.feature.rms(y=y)),
        #np.ndarray.flatten(librosa.feature.spectral_centroid(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.spectral_contrast(y=y,sr=sr,fmin=100)),
        np.ndarray.flatten(librosa.feature.spectral_bandwidth(y=y,sr=sr)),
        np.ndarray.flatten(librosa.feature.spectral_flatness(y=y)),
        np.ndarray.flatten(librosa.feature.spectral_rolloff(y=y,sr=sr))

    ))


print('extracting librosa features for train_x:')
train_x_features = np.array([extract_librosa_features(sample, sr)
                             for sample in tqdm(train_x)])
train_x_features = preprocessing.StandardScaler().fit_transform(train_x_features)

print('extracting librosa features for test_x:')
test_x_features = np.array([extract_librosa_features(sample, sr)
                            for sample in tqdm(test_x)])
test_x_features = preprocessing.StandardScaler().fit_transform(test_x_features)

print('#######Gradient Boosting on Librosa Features########')
GBC = ensemble.HistGradientBoostingClassifier(learning_rate=.15, max_iter=40,verbose=True)
start= time.time()
GBC.fit(train_x_features,train_y)
interval1 = time.time()
print(f'GBC model fit time: {interval1-start:.2f} s')
print(f'training accuracy: {GBC.score(train_x_features,train_y)}')
interval2 = time.time()
print(f'train predict time: {interval2-interval1:.2f} s')
print(f'test accuracy: {GBC.score(test_x_features, test_y):.4f}')
interval3 = time.time()
print(f'test predict time: {interval3-interval2:.2f} s')

extracting librosa features for train_x:


100%|███████████| 11000/11000 [08:24<00:00, 21.79it/s]


extracting librosa features for test_x:


100%|█████████████| 1000/1000 [00:48<00:00, 20.47it/s]


#######Gradient Boosting on Librosa Features########
Binning 0.112 GB of training data: 4.858 s
Binning 0.012 GB of validation data: 0.161 s
Fitting gradient boosted rounds:
[1/40] 5 trees, 155 leaves (31 on avg), max depth = 13, train loss: 1.40931, val loss: 1.46626, in 1.238s
[2/40] 5 trees, 155 leaves (31 on avg), max depth = 20, train loss: 1.26186, val loss: 1.36594, in 1.300s
[3/40] 5 trees, 155 leaves (31 on avg), max depth = 13, train loss: 1.14750, val loss: 1.28630, in 1.327s
[4/40] 5 trees, 155 leaves (31 on avg), max depth = 19, train loss: 1.04932, val loss: 1.21145, in 1.334s
[5/40] 5 trees, 155 leaves (31 on avg), max depth = 17, train loss: 0.96543, val loss: 1.15143, in 1.441s
[6/40] 5 trees, 155 leaves (31 on avg), max depth = 19, train loss: 0.89414, val loss: 1.09884, in 1.331s
[7/40] 5 trees, 155 leaves (31 on avg), max depth = 14, train loss: 0.83199, val loss: 1.06002, in 1.303s
[8/40] 5 trees, 155 leaves (31 on avg), max depth = 16, train loss: 0.77838, val los